In [1]:
from ultralytics import YOLO
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


model=YOLO("best.pt")
base=python.BaseOptions("hand_landmarker.task")

options = vision.HandLandmarkerOptions(base_options=base, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # اكتشاف اليد أو الشخص بواسطة YOLO
    results = model(frame, verbose=False)
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # قص منطقة اليد (ROI)
            hand_img = frame[y1:y2, x1:x2]
            if hand_img.size == 0: continue
            
            # تحويل الصورة لتناسب MediaPipe Tasks
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(hand_img, cv2.COLOR_BGR2RGB))
            
            # استخراج النقاط (Landmarks)
            detection_result = detector.detect(mp_image)
            
            # معالجة النتائج ورسمها
            if detection_result.hand_landmarks:
                for hand_landmarks in detection_result.hand_landmarks:
                    # هنا يمكنك الوصول للنقاط: hand_landmarks[0].x , hand_landmarks[0].y
                    # ملاحظة: الإحداثيات هنا داخل المربع المقصوص
                    cv2.putText(frame, "Hand Detected", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    cv2.imshow('YOLO + MediaPipe Tasks', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

yolo_model = YOLO("best.pt")

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    results = yolo_model(frame, verbose=False)
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            label = yolo_model.names[cls_id]
            confidence = box.conf[0]
            
            hand_roi = frame[max(0, y1):y2, max(0, x1):x2]
            if hand_roi.size == 0: continue
            
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(hand_roi, cv2.COLOR_BGR2RGB))
            
            detection_result = detector.detect(mp_image)
            
            if detection_result.hand_landmarks:
                for hand_landmarks in detection_result.hand_landmarks:
                    for lm in hand_landmarks:
                        cx = int(lm.x * (x2 - x1)) + x1
                        cy = int(lm.y * (y2 - y1)) + y1
                        cv2.circle(frame, (cx, cy), 3, (0, 255, 255), -1)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow('YOLO 120 Classes + MP Tasks', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()